# Runoff contribution


In [ ]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr


# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from src.utils import load_config, weighted_mean_by_lat
from src.plotting import base_layout, geo_style, add_region_boundary_ruler, add_panel_label, save_figure

cfg = load_config()

## Data

In [ ]:
common_period = slice("2001-01-01", "2025-12-31")

# plot elements
main_basins = gpd.read_file(cfg["dir"]["data"] + "/hydroatlas/hybas_sa_lev08_v1c_dissolved.shp")
main_basins["geometry"] = main_basins["geometry"].simplify(0.02, preserve_topology=True)

# polygons / shapes
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
#hydroatlas_shape = hydroatlas_shape[hydroatlas_shape.glacier_cover_RGI60 > 0]
hydroatlas_shape['basin_cenlat_rounded'] = hydroatlas_shape['basin_cenlat'].round(0)

# total runoff in mm/day
hydroatlas_runoff = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc", chunks = "auto").q_mm_sim
hydroatlas_runoff = hydroatlas_runoff.sel(date=common_period).rename("total_runoff")

# glacier melt in m3/s -> mm/day
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=hydroatlas_shape.index, fill_value=0)
hydroatlas_gmelt = hydroatlas_gmelt.sel(date=common_period)
hydroatlas_gmelt = (hydroatlas_gmelt * 1000 * 86400) / (hydroatlas_shape.basin_area.values * 1e6)[:, None]

# runoff 
ds = xr.merge([hydroatlas_runoff, hydroatlas_gmelt], join='outer').resample(date="MS").sum().load()
ds["gmelt_ratio"] = ds.glacier_melt / ds.total_runoff

ds_annual = ds.resample(date="YS").sum().mean(dim="date")
ds_annual["gmelt_ratio"] = ds_annual.glacier_melt / ds_annual.total_runoff # rewrite

ds_summer = ds.sel(date=ds.date.dt.month.isin([12, 1, 2])).resample(date="YS").sum().mean(dim = "date")
ds_summer["gmelt_ratio"] = ds_summer.glacier_melt / ds_summer.total_runoff # rewrite

ds_monthly = ds["gmelt_ratio"].groupby("date.month").mean()
ds_monthly = ds_monthly.isel(month=ds_monthly.argmax(dim="month"))

In [ ]:
# lat processing
hydroatlas_shape = hydroatlas_shape.join([
    ds_annual.total_runoff.rename("runoff_annual").to_pandas(),
    ds_annual.glacier_melt.rename("gmelt_annual").to_pandas(),
    ds_annual.gmelt_ratio.rename("gmelt_ratio_annual").to_pandas(),
    ds_summer.gmelt_ratio.rename("gmelt_ratio_summer").to_pandas(),
    ds_monthly.rename("gmelt_ratio_monthly").to_pandas(),
])

hydroatlas_shape = hydroatlas_shape.sort_values('runoff_annual')
hydroatlas_shape_lat = weighted_mean_by_lat(hydroatlas_shape) 

## Plot (MS)

In [ ]:
# Bivariate palette (4x4): cols = runoff (low→high), rows = gmelt (low→high)
n_classes = 4

bivariate_colors = [
    ["#ebebeb", "#83c1dd", "#428fc6", "#2269b0"],  # low gmelt
    ["#e7dc7b", "#84bd7b", "#41907e", "#236b88"],  # mid-low gmelt
    ["#e2c33b", "#8bb539", "#48923f", "#287151"],  # mid-high gmelt
    ["#deab1d", "#9ca71e", "#5c9622", "#388030"],  # high gmelt
]

runoff_bins = pd.qcut(hydroatlas_shape['runoff_annual'], n_classes, labels=False, duplicates='drop')
gmelt_bins  = pd.qcut(hydroatlas_shape['gmelt_annual'],  n_classes, labels=False, duplicates='drop')
point_colors = [bivariate_colors[int(g)][int(r)] for g, r in zip(gmelt_bins, runoff_bins)]

# Add city markers and labels
cities = {
    "  Huaraz":            (-9.53, -77.53),
    "  Cusco":            (-13.52, -71.97),
    "  La Paz":           (-16.50, -68.15),
    "  Calama":           (-22.46, -68.93),
    "  Vallenar":         (-28.50, -70.76),
    "  Mendoza":          (-32.70, -68.85),
    "  Santiago":         (-33.60, -70.65),
    "  Talca":            (-35.43, -71.67),
    "  Temuco":           (-38.74, -72.59),
    "  Bariloche":        (-41.13, -71.31),
    "  Coyhaique":        (-45.57, -72.07),
    "  El Chaltén":       (-49.33, -72.88),
    "  El Calafate":      (-50.34, -72.28),
    "  Puerto Natales":   (-51.73, -72.49),
    "  Tierra<br>del Fuego": (-54.80, -68.30),
}

In [ ]:
# Create subplot structure
fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    horizontal_spacing=0.01,
    column_widths=[0.6, 0.4],
    specs=[[{"type": "scattergeo"}, {"type": "xy"}]]
)

# Plot Nº1: Bivariate map
fig.add_trace(
    go.Scattergeo(
        lon=hydroatlas_shape['basin_cenlon'],
        lat=hydroatlas_shape['basin_cenlat'],
        mode='markers',
        marker=dict(
            size=5,
            color=point_colors,
            opacity=0.95,
            line=dict(width=0.1, color='white')
        ),
        showlegend=False
    ),
    row=1, col=1
)

# Add river shapes
fig.add_trace(go.Choropleth(geojson=eval(main_basins['geometry'].to_json()), locations=main_basins.index, 
                            z=main_basins.NEXT_DOWN, colorscale=["rgba(255,255,255,0.0001)", "rgba(255,255,255,0.0001)"], 
                            marker_line_color = "rgb(210, 226, 230)", showscale=False, marker_line_width=0.3), 
                row=1, col=1)


# Add city labels
city_line_lat, city_line_lon = [], []
city_label_lat, city_label_lon = [], []
dx = 1.8
for name, (lat, lon) in cities.items():
    city_line_lat += [lat, lat, None]
    city_line_lon += [lon, lon + dx, None]
    city_label_lat.append(lat)
    city_label_lon.append(lon + dx)

# Leader lines (no arrowhead)
fig.add_trace(
    go.Scattergeo(
        lat=city_line_lat,
        lon=city_line_lon,
        mode="lines",
        line=dict(width=0.7, color="#5e5e5e"),
        showlegend=False
    ), row=1, col=1)

# City markers
fig.add_trace(
    go.Scattergeo(
        lat=[lat for lat, lon in cities.values()],
        lon=[lon for lat, lon in cities.values()],
        mode="markers",
        marker=dict(size=5, symbol="circle", color="#5e5e5e"),
        showlegend=False
    ), row=1, col=1)

# City name labels (offset from marker)
fig.add_trace(
    go.Scattergeo(
        lat=city_label_lat,
        lon=city_label_lon,
        text=list(cities.keys()),
        mode="text",
        textposition="middle right",
        textfont=dict(size=9, color="#5e5e5e"),
        showlegend=False
    ), row=1, col=1)


# Update geography settings
fig.update_geos(**geo_style([-87, -62], countrycolor="rgba(0,0,0,0.6)", countrywidth=0.1), row=1, col=1)

# Add region-boundary ruler on the left of the map
add_region_boundary_ruler(
    fig, cfg["regions"], boundary_lats=[-7.2, -20, -30, -35, -45, -52, -55.8],
    x=0.06, font_size=12,
)

# Bivariate legend (4x4 grid)
lx0, ly0 = 0.15, 0.60
cell_w, cell_h = 0.035, 0.035
for r in range(n_classes):      # runoff: 0=low → 3=high (left to right)
    for g in range(n_classes):  # gmelt:  0=low → 3=high (bottom to top)
        fig.add_shape(
            type="rect",
            x0=lx0 + r * cell_w,       x1=lx0 + (r + 1) * cell_w,
            y0=ly0 + g * cell_h,        y1=ly0 + (g + 1) * cell_h,
            xref="paper", yref="paper",
            fillcolor=bivariate_colors[g][r],
            line=dict(width=0.5, color="white"),
        )

fig.add_annotation(text="Total runoff →", x=lx0 + 1.8 * cell_w, y=ly0 - 0.012,
                   xref="paper", yref="paper", showarrow=False,
                   font=dict(size=11), xanchor="center")
fig.add_annotation(text="Glacier melt →", x=lx0 - 0.012, y=ly0 + 1.8 * cell_h,
                   xref="paper", yref="paper", showarrow=False,
                   font=dict(size=11), xanchor="center", textangle=-90)



# Plot Nº2: Glacier melt to runoff ratio by latitude
fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_monthly']*100,
        y=hydroatlas_shape_lat.index,
        name="Peak month",
        marker_color= "#f5c211",
        orientation='h',
        opacity=1,
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_summer']*100,
        y=hydroatlas_shape_lat.index,
        name="Summer",
        marker_color="#73a539",
        orientation='h',
        opacity=1,
    ), 
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=hydroatlas_shape_lat['gmelt_ratio_annual']*100,
        y=hydroatlas_shape_lat.index,
        name="Annual",
        marker_color= "#457bb0",
        orientation='h',
        opacity=1,
    ),
    row=1, col=2
)


fig.update_yaxes(title_standoff=2, ticksuffix="ºS", dtick = 5, range=(-56, -7), side="right", row=1, col=2)
fig.update_xaxes(title="Contribution rel. runoff(%)", title_standoff = 2,  range = (0, 70), dtick=10, row=1, col=2)

# Update axes + annotations
fig.update_yaxes(ticks="inside")
add_panel_label(fig, "a)", x=0.57, y=0.995)
add_panel_label(fig, "b)", x=0.99, y=0.995)

# Update layout
fig.update_layout(**base_layout(
    autosize=False, 
    barmode="overlay",
    bargap=0.1,
    margin=dict(l=10, r=50, b=40, t=10),
    legend=dict(orientation="v", y=0.94, x=0.80, bgcolor="rgba(255,255,255,0)"),
    width=630, 
    height=700, 
    boxmode='group',
    font=dict(size=13),
))

save_figure(fig, "figures/fig_runoff_contribution.svg", width=630, height=700)
fig.show()

## Plot (slides)

In [ ]:
# andeanGC data
metadata_andeangc   = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "AndeanGC_metadata.csv", index_col=0)

runoff_andeangc_sim = xr.open_dataset("data/hydro/runoff_sim_andeangc.nc").q_mm_sim
runoff_andeangc_sim = runoff_andeangc_sim.resample(date="YS").sum().mean(dim="date")
runoff_andeangc_sim = runoff_andeangc_sim.to_series()

metadata_andeangc = metadata_andeangc.join(runoff_andeangc_sim.rename("runoff_annual"))
metadata_andeangc = metadata_andeangc.dropna()